# DL_project: Reproducible Alzheimer MRI Multi-Foundation Pipeline

이 노트북은 최종 프로젝트를 재현하기 위한 참고용 노트북이다.

기본 실행은 기존 결과물을 로드해서 빠르게 확인하는 구조이다. feature cache 재생성, adapter 재학습, SAM/Occlusion 재생성, LM Studio LLM report 재생성은 상단 flag를 `True`로 바꿔서 실행한다.

## 1. 환경 점검

In [ ]:
import os
import sys
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except Exception as e:
    torch = None
    print("torch import 실패:", repr(e))

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)

if torch is not None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Python executable:", sys.executable)
    print("Python version:", sys.version)
    print("torch version:", torch.__version__)
    print("torch.version.cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA GPU not available")
    print("device:", device)
else:
    device = None

## 2. 경로와 실행 옵션

In [ ]:
PROJECT_DIR = Path(r"C:\Users\user\alzheimer")
DATA_DIR = Path(r"C:\Users\user\Desktop\alzheimer_dataset")

FOUNDATION_DIR = PROJECT_DIR / "patient_level_stage1_foundation"
FOUNDATION_CACHE_DIR = FOUNDATION_DIR / "cache"
FOUNDATION_CHECKPOINT_DIR = FOUNDATION_DIR / "checkpoints"
PIPELINE_DIR = PROJECT_DIR / "sam_occlusion_llm_pipeline"
FINAL_PPT_ASSETS = PROJECT_DIR / "final_ppt_assets"

FEATURE_CACHE_PATH = FOUNDATION_CACHE_DIR / "non_vs_demented_biomedclip_all_original_features.pt"
PATIENT_TABLE_PATH = FOUNDATION_DIR / "non_vs_demented_biomedclip_patient_table.csv"
MANIFEST_PATH = FOUNDATION_DIR / "non_vs_demented_biomedclip_all_images_manifest.csv"
MODEL_TABLE_PATH = PROJECT_DIR / "presentation_assets" / "final_model_comparison_table.csv"
OOF_PRED_PATH = PIPELINE_DIR / "adapter_probe_oof_patient_predictions.csv"
OOF_METRICS_PATH = PIPELINE_DIR / "adapter_probe_oof_metrics.json"
REPORT_INDEX_PATH = PIPELINE_DIR / "sam_occlusion_llm_report_index.csv"

# 기본값은 빠른 재현이다. 무거운 단계는 필요할 때만 True로 바꾼다.
RUN_FEATURE_CACHE = False
RUN_TRAIN_ADAPTER = False
RUN_SAM_OCCLUSION = False
RUN_LLM_REPORT = False

MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
NUM_WORKERS = 0
PIN_MEMORY = True
USE_AMP = True
FEATURE_BATCH_SIZE = 128

N_SPLITS = 5
INNER_VAL_RATIO = 0.15
PROBE_BATCH_SIZE = 512
PROBE_EPOCHS = 20
PROBE_PATIENCE = 4
PROBE_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-4
ADAPTER_HIDDEN_DIM = 128
ADAPTER_DROPOUT = 0.2
TARGET_SENSITIVITY = 0.85

REPRO_OUTPUT_DIR = PROJECT_DIR / "DL_project_repro_outputs"
REPRO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    PROJECT_DIR,
    PATIENT_TABLE_PATH,
    MANIFEST_PATH,
    FEATURE_CACHE_PATH,
    MODEL_TABLE_PATH,
    OOF_PRED_PATH,
    OOF_METRICS_PATH,
    REPORT_INDEX_PATH,
]

for path in required_paths:
    print(f"{path} ->", "OK" if path.exists() else "MISSING")

## 3. 데이터 인덱싱과 Patient-Level Label 정의

In [ ]:
CLASS_NAMES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
DEMENTED_CLASSES = {"VeryMildDemented", "MildDemented", "ModerateDemented"}

def extract_patient_id(path_or_name):
    text = str(path_or_name)
    match = re.search(r"(OAS1_\d+)", text)
    if match:
        return match.group(1)
    return Path(text).stem.split("_MR")[0]

def target_from_class(class_name):
    if class_name == "NonDemented":
        return 0
    if class_name in DEMENTED_CLASSES:
        return 1
    raise ValueError(f"unknown class: {class_name}")

def build_image_manifest(data_dir=DATA_DIR):
    rows = []
    for class_name in CLASS_NAMES:
        class_dir = Path(data_dir) / class_name
        if not class_dir.exists():
            continue
        for image_path in class_dir.rglob("*"):
            if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                rows.append({
                    "image_path": str(image_path),
                    "class_name": class_name,
                    "target": target_from_class(class_name),
                    "patient_id": extract_patient_id(image_path.name),
                })
    return pd.DataFrame(rows)

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)
    print("기존 manifest 로드:", manifest.shape)
elif DATA_DIR.exists():
    manifest = build_image_manifest(DATA_DIR)
    print("새 manifest 생성:", manifest.shape)
else:
    manifest = pd.DataFrame()
    print("DATA_DIR가 없어 manifest를 만들지 않았습니다.")

if PATIENT_TABLE_PATH.exists():
    patient_table = pd.read_csv(PATIENT_TABLE_PATH)
elif not manifest.empty:
    patient_table = (
        manifest.groupby("patient_id")
        .agg(class_name=("class_name", "first"), target=("target", "first"), n_images=("image_path", "count"))
        .reset_index()
    )
else:
    patient_table = pd.DataFrame()

display(patient_table.head())
if not patient_table.empty:
    display(patient_table["class_name"].value_counts())
    display(patient_table["target"].value_counts().rename({0: "NonDemented", 1: "Demented"}))

## 4. Patient-Level 5-Fold Split 재현

In [ ]:
from sklearn.model_selection import StratifiedKFold

def make_patient_folds(patient_df, n_splits=5, seed=42):
    df = patient_df.copy().reset_index(drop=True)
    df["fold"] = -1
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (_, test_idx) in enumerate(splitter.split(df["patient_id"], df["target"]), start=1):
        df.loc[test_idx, "fold"] = fold
    return df

if not patient_table.empty:
    folds = make_patient_folds(patient_table, n_splits=5, seed=SEED)
    display(pd.crosstab(folds["fold"], folds["target"]).rename(columns={0: "NonDemented", 1: "Demented"}))

## 5. BiomedCLIP Feature Cache 로드 또는 재생성

In [ ]:
# 기본은 기존 cache 로드이다. 완전 재현이 필요하면 RUN_FEATURE_CACHE=True로 바꾸고 실행한다.
# 중요: feature cache 단계에는 augmentation, sampler, shuffle을 넣지 않는다.
# cache는 같은 원본 이미지가 항상 같은 feature로 매핑된다는 전제가 있어야 하기 때문이다.

def load_feature_cache(path=FEATURE_CACHE_PATH):
    if path.exists() and torch is not None:
        cached = torch.load(path, map_location="cpu", weights_only=False)
        print("feature cache keys:", cached.keys())
        print("features:", tuple(cached["features"].shape))
        print("labels:", tuple(cached["labels"].shape))
        print("model:", cached.get("model_name"))
        print("normalized:", cached.get("normalized"))
        return cached
    print("feature cache를 찾지 못했거나 torch를 사용할 수 없습니다.")
    return None

cache = load_feature_cache()

if RUN_FEATURE_CACHE:
    if torch is None:
        raise RuntimeError("torch가 필요합니다.")

    import open_clip
    import torch.nn.functional as F
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader

    class FoundationImageDataset(Dataset):
        def __init__(self, dataframe, preprocess):
            self.df = dataframe.reset_index(drop=True).copy()
            self.preprocess = preprocess

        def __len__(self):
            return len(self.df)

        def __getitem__(self, index):
            row = self.df.iloc[index]
            image = Image.open(row["image_path"]).convert("RGB")
            image = self.preprocess(image)
            return (
                image,
                torch.tensor(int(row["target"]), dtype=torch.long),
                row["patient_id"],
                row["image_path"],
                row["class_name"],
            )

    print("BiomedCLIP 로드 중...")
    biomed_model, preprocess = open_clip.create_model_from_pretrained(MODEL_NAME)
    biomed_model = biomed_model.to(device)
    biomed_model.eval()
    assert next(biomed_model.parameters()).device.type == device.type

    dataset = FoundationImageDataset(manifest, preprocess)
    loader = DataLoader(
        dataset,
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        sampler=None,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    features, labels = [], []
    patient_ids, image_paths, class_names = [], [], []
    start = time.time()
    seen = 0

    with torch.inference_mode():
        for batch_idx, batch in enumerate(loader, start=1):
            images, batch_labels, batch_patient_ids, batch_image_paths, batch_class_names = batch
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
                batch_features = biomed_model.encode_image(images)
                batch_features = F.normalize(batch_features.float(), dim=-1)

            features.append(batch_features.cpu())
            labels.append(batch_labels.cpu())
            patient_ids.extend(list(batch_patient_ids))
            image_paths.extend(list(batch_image_paths))
            class_names.extend(list(batch_class_names))

            seen += images.size(0)
            elapsed = max(time.time() - start, 1e-6)
            speed = seen / elapsed
            eta_min = (len(dataset) - seen) / max(speed, 1e-6) / 60
            if batch_idx == 1 or batch_idx % 20 == 0 or seen == len(dataset):
                print(f"{seen:,}/{len(dataset):,} images | {speed:.1f} images/s | ETA {eta_min:.1f} min")

    cache = {
        "features": torch.cat(features, dim=0),
        "labels": torch.cat(labels, dim=0),
        "patient_ids": patient_ids,
        "image_paths": image_paths,
        "class_names": class_names,
        "model_name": MODEL_NAME,
        "normalized": True,
    }
    torch.save(cache, FEATURE_CACHE_PATH)
    print("feature cache saved:", FEATURE_CACHE_PATH)

## 6. Adapter Probe 학습 구조

In [ ]:
# 이 셀은 최종 classifier인 BiomedCLIP frozen feature + adapter probe를 재현한다.
# RUN_TRAIN_ADAPTER=False일 때는 구조만 확인하고, True일 때는 5-fold 학습을 다시 수행한다.

if torch is not None:
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        average_precision_score,
        confusion_matrix,
    )

    class CachedFeatureDataset(Dataset):
        def __init__(self, cached, indices):
            self.features = cached["features"]
            self.labels = cached["labels"].long()
            self.patient_ids = cached["patient_ids"]
            self.indices = np.asarray(indices, dtype=np.int64)

        def __len__(self):
            return len(self.indices)

        def __getitem__(self, item):
            index = int(self.indices[item])
            return self.features[index].float(), self.labels[index], self.patient_ids[index]

    class AdapterProbe(nn.Module):
        def __init__(self, input_dim=512, hidden_dim=128, dropout=0.2):
            super().__init__()
            self.norm = nn.LayerNorm(input_dim)
            self.adapter = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, input_dim),
            )
            self.classifier = nn.Linear(input_dim, 2)

        def forward(self, features):
            normalized = self.norm(features)
            adapted = normalized + self.adapter(normalized)
            return self.classifier(adapted)

    def count_trainable_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    def indices_for_patient_ids(cached, patient_ids):
        patient_set = set(patient_ids)
        return [i for i, pid in enumerate(cached["patient_ids"]) if pid in patient_set]

    def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
        df = pd.DataFrame({
            "patient_id": patient_ids,
            "target": labels,
            "probability": probabilities,
        })
        patient_pred = (
            df.groupby("patient_id")
            .agg(target=("target", "first"), probability=("probability", "mean"))
            .reset_index()
        )
        patient_pred["pred_label"] = (patient_pred["probability"] >= threshold).astype(int)
        return patient_pred

    def calculate_metrics(y_true, y_prob, threshold=0.5):
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        y_pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "sensitivity": recall_score(y_true, y_pred, zero_division=0),
            "specificity": tn / max(tn + fp, 1),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "auroc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
            "auprc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        }

    def choose_sensitivity_first_threshold(y_true, y_prob, target_sensitivity=TARGET_SENSITIVITY):
        thresholds = np.linspace(0.05, 0.95, 181)
        rows = []
        for threshold in thresholds:
            metrics = calculate_metrics(y_true, y_prob, threshold)
            rows.append({"threshold": threshold, **metrics})
        grid = pd.DataFrame(rows)
        candidates = grid[grid["sensitivity"] >= target_sensitivity].copy()
        if len(candidates):
            best = candidates.sort_values(["specificity", "f1"], ascending=False).iloc[0]
        else:
            best = grid.sort_values(["sensitivity", "f1"], ascending=False).iloc[0]
        return float(best["threshold"])

    def evaluate_model(model, loader):
        model.eval()
        patient_ids, labels_all, probs_all = [], [], []
        with torch.inference_mode():
            for features, labels, batch_patient_ids in loader:
                features = features.to(device, non_blocking=True)
                logits = model(features)
                probs = logits.softmax(dim=1)[:, 1].detach().cpu().numpy()
                patient_ids.extend(list(batch_patient_ids))
                labels_all.extend(labels.numpy().tolist())
                probs_all.extend(probs.tolist())
        return aggregate_patient_predictions(patient_ids, labels_all, probs_all, threshold=0.5)

    demo_model = AdapterProbe(input_dim=512, hidden_dim=ADAPTER_HIDDEN_DIM, dropout=ADAPTER_DROPOUT)
    print(demo_model)
    print("trainable params:", count_trainable_parameters(demo_model))

if RUN_TRAIN_ADAPTER:
    assert cache is not None, "feature cache가 필요합니다."
    assert torch is not None, "torch가 필요합니다."

    feature_dim = int(cache["features"].shape[1])
    all_oof_rows = []
    fold_results = []

    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    patient_df = patient_table.reset_index(drop=True).copy()

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(patient_df["patient_id"], patient_df["target"]),
        start=1,
    ):
        seed_everything(SEED + fold)
        outer_train = patient_df.iloc[train_idx].copy()
        outer_test = patient_df.iloc[test_idx].copy()

        inner_train, inner_val = train_test_split(
            outer_train,
            test_size=INNER_VAL_RATIO,
            random_state=SEED + fold,
            stratify=outer_train["target"],
        )

        train_indices = indices_for_patient_ids(cache, inner_train["patient_id"])
        val_indices = indices_for_patient_ids(cache, inner_val["patient_id"])
        test_indices = indices_for_patient_ids(cache, outer_test["patient_id"])

        train_loader = DataLoader(
            CachedFeatureDataset(cache, train_indices),
            batch_size=PROBE_BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        val_loader = DataLoader(
            CachedFeatureDataset(cache, val_indices),
            batch_size=PROBE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )
        test_loader = DataLoader(
            CachedFeatureDataset(cache, test_indices),
            batch_size=PROBE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

        model = AdapterProbe(feature_dim, ADAPTER_HIDDEN_DIM, ADAPTER_DROPOUT).to(device)
        class_counts = inner_train["target"].value_counts().reindex([0, 1]).fillna(1).to_numpy()
        class_weight = torch.tensor(class_counts.sum() / (2 * class_counts), dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=PROBE_LR, weight_decay=PROBE_WEIGHT_DECAY)

        best_state = None
        best_val = -np.inf
        early = 0

        print(f"fold {fold}: train patients={len(inner_train)}, val={len(inner_val)}, test={len(outer_test)}")
        for epoch in range(1, PROBE_EPOCHS + 1):
            model.train()
            loss_sum, total = 0.0, 0
            for features, labels, _ in train_loader:
                features = features.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                logits = model(features)
                loss = criterion(logits, labels)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                loss_sum += loss.item() * labels.size(0)
                total += labels.size(0)

            val_pred = evaluate_model(model, val_loader)
            val_metrics = calculate_metrics(val_pred["target"], val_pred["probability"], threshold=0.5)
            score = val_metrics["auroc"]
            print(f"Epoch {epoch:02d} | loss {loss_sum/max(total,1):.4f} | val AUROC {score:.4f} AUPRC {val_metrics['auprc']:.4f}")

            if score > best_val + 1e-4:
                best_val = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                early = 0
            else:
                early += 1
                if early >= PROBE_PATIENCE:
                    print("early stopping")
                    break

        model.load_state_dict(best_state)
        model = model.to(device)

        val_pred = evaluate_model(model, val_loader)
        threshold = choose_sensitivity_first_threshold(
            val_pred["target"],
            val_pred["probability"],
            target_sensitivity=TARGET_SENSITIVITY,
        )
        test_pred = evaluate_model(model, test_loader)
        test_pred["fold"] = fold
        test_pred["threshold"] = threshold
        test_pred["pred_label"] = (test_pred["probability"] >= threshold).astype(int)
        all_oof_rows.append(test_pred)

        metrics = calculate_metrics(test_pred["target"], test_pred["probability"], threshold=threshold)
        fold_results.append({"fold": fold, "threshold": threshold, **metrics})
        torch.save(
            {"model_state_dict": best_state, "feature_dim": feature_dim, "threshold": threshold},
            REPRO_OUTPUT_DIR / f"adapter_probe_fold{fold}.pt",
        )

    repro_oof = pd.concat(all_oof_rows, ignore_index=True)
    repro_results = pd.DataFrame(fold_results)
    repro_oof.to_csv(REPRO_OUTPUT_DIR / "adapter_probe_oof_predictions.csv", index=False, encoding="utf-8-sig")
    repro_results.to_csv(REPRO_OUTPUT_DIR / "adapter_probe_fold_results.csv", index=False, encoding="utf-8-sig")
    display(repro_results)
    display(repro_oof.head())
else:
    print("RUN_TRAIN_ADAPTER=False: 기존 저장 결과를 로드하는 방식으로 진행합니다.")

## 7. 최종 OOF 결과 로드와 평가

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

oof_pred = pd.read_csv(OOF_PRED_PATH)
with OOF_METRICS_PATH.open("r", encoding="utf-8") as f:
    oof_metrics = json.load(f)

print(json.dumps(oof_metrics, indent=2, ensure_ascii=False))

metric_order = ["sensitivity", "specificity", "f1", "macro_f1", "auroc", "auprc"]
metric_df = pd.DataFrame({
    "metric": metric_order,
    "value": [oof_metrics.get(m) for m in metric_order],
})
display(metric_df)

cmat = confusion_matrix(oof_pred["target"], oof_pred["pred_label"], labels=[0, 1])
print(cmat)
print(classification_report(oof_pred["target"], oof_pred["pred_label"], target_names=["NonDemented", "Demented"]))

fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cmat, cmap="Blues")
ax.set_title("BiomedCLIP Adapter Probe Confusion Matrix")
ax.set_xticks([0, 1], ["Pred NonDemented", "Pred Demented"])
ax.set_yticks([0, 1], ["True NonDemented", "True Demented"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cmat[i, j]), ha="center", va="center", fontsize=18)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## 8. 모델 비교표 로드

In [ ]:
model_table = pd.read_csv(MODEL_TABLE_PATH)
display(model_table)

metric_cols = ["Sensitivity", "Specificity", "F1", "AUROC", "AUPRC"]
plot_df = model_table.copy()
for col in metric_cols:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

ax = plot_df.set_index("Method")[metric_cols].plot(kind="bar", figsize=(13, 5), width=0.8)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison")
ax.legend(ncol=5, bbox_to_anchor=(0.5, 1.18), loc="upper center")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 9. SAM + Occlusion + LLM 결과 로드

In [ ]:
from IPython.display import Image, display, Markdown

report_index = pd.read_csv(REPORT_INDEX_PATH)
display(report_index)

preview_path = PIPELINE_DIR / "demo_assets" / "sam_occlusion_classifier_preview.png"
if preview_path.exists():
    display(Image(filename=str(preview_path)))

reports_dir = PIPELINE_DIR / "llm_reports"
report_files = sorted(reports_dir.glob("*_llm_report.md"))
print("LLM report files:")
for path in report_files:
    print("-", path)

if report_files:
    sample_report = report_files[0]
    display(Markdown(sample_report.read_text(encoding="utf-8")))

## 10. LM Studio LLM Report 재생성 옵션

In [ ]:
# LM Studio local server 예시:
# base_url = "http://127.0.0.1:1234/v1"
# model = "google/gemma-4-e4b"
#
# RUN_LLM_REPORT=True일 때만 prompts를 다시 보내 report를 재생성한다.

if RUN_LLM_REPORT:
    import requests

    BASE_URL = "http://127.0.0.1:1234/v1"
    MODEL_ID = "google/gemma-4-e4b"
    PROMPT_DIR = PIPELINE_DIR / "llm_prompts"
    REPORT_DIR = PIPELINE_DIR / "llm_reports"
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    for prompt_path in sorted(PROMPT_DIR.glob("*_prompt.md")):
        prompt_text = prompt_path.read_text(encoding="utf-8")
        response = requests.post(
            f"{BASE_URL}/chat/completions",
            json={
                "model": MODEL_ID,
                "messages": [
                    {"role": "system", "content": "You summarize research screening outputs in Korean. Do not diagnose."},
                    {"role": "user", "content": prompt_text},
                ],
                "temperature": 0.2,
            },
            timeout=120,
        )
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
        out_path = REPORT_DIR / prompt_path.name.replace("_prompt.md", "_llm_report.md")
        out_path.write_text(content, encoding="utf-8")
        print("saved:", out_path)
else:
    print("RUN_LLM_REPORT=False: 기존 LLM report를 사용합니다.")

## 11. 최종 산출물 Index

In [ ]:
important_paths = {
    "DL_project markdown": PROJECT_DIR / "DL_project.md",
    "DL_project notebook": PROJECT_DIR / "DL_project.ipynb",
    "PPT assets": FINAL_PPT_ASSETS,
    "Feature cache": FEATURE_CACHE_PATH,
    "OOF predictions": OOF_PRED_PATH,
    "OOF metrics": OOF_METRICS_PATH,
    "LLM reports": PIPELINE_DIR / "llm_reports",
    "SAM/Occlusion assets": PIPELINE_DIR / "demo_assets",
}

index_df = pd.DataFrame([
    {"artifact": name, "path": str(path), "exists": path.exists()}
    for name, path in important_paths.items()
])
display(index_df)